# GRAM Recurrent Qwen SVGD Robustness Launcher

Fresh-runtime launcher for the current projected-kernel robustness experiment.

Required Colab secrets:

- `GH_TOKEN`: GitHub token with access to `mshapiro123/recurrent-qwen-svgd`
- `HF_TOKEN`: Hugging Face token

Required Drive checkpoints are restored by filename:

- `phase1_step_150.pt` -> `outputs/qwen_0_5b_phase1_a100_beta008_continue_150/phase1_step_150.pt`
- `phase2_step_25.pt` -> `outputs/qwen_0_5b_phase2_svgd_smoke25/phase2_step_25.pt`


In [1]:
import os, subprocess, sys
from pathlib import Path

print(sys.version)
subprocess.run(['nvidia-smi'], check=False)


3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:
%cd /content

import os, subprocess
from pathlib import Path
from google.colab import userdata

REPO = 'mshapiro123/recurrent-qwen-svgd'
PROJECT = Path('/content/recurrent-qwen-svgd')

gh = userdata.get('GH_TOKEN')
assert gh, 'Missing GH_TOKEN in Colab Secrets.'

askpass = Path('/tmp/git-askpass.sh')
askpass.write_text(
    '#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token" ;;\n*) echo "$GH_TOKEN" ;;\nesac\n',
    encoding='utf-8',
)
askpass.chmod(0o700)

env = os.environ.copy()
env['GH_TOKEN'] = gh
env['GIT_ASKPASS'] = str(askpass)
env['GIT_TERMINAL_PROMPT'] = '0'

if PROJECT.exists():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], env=env, check=True)
else:
    subprocess.run(['git', 'clone', f'https://github.com/{REPO}.git', str(PROJECT)], env=env, check=True)

%cd /content/recurrent-qwen-svgd
!git log --oneline -5


/content
/content/recurrent-qwen-svgd
8f890fb (HEAD -> main, origin/main, origin/HEAD) Add Colab robustness launcher notebook
9e2b3e6 Record projected kernel diagnostics
d2cd499 Add projected SVGD kernel option
79ea98b Record SVGD diagnostic stats
1453667 Record v2 smoke results and add summarizer


In [3]:
%cd /content/recurrent-qwen-svgd
!python -m pip install -q -r requirements.txt
!python -m pip show transformers torch | sed -n '1,30p'


/content/recurrent-qwen-svgd
Name: transformers
Version: 5.10.2
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: peft, sentence-transformers
---
Name: torch
Version: 2.11.0+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires

In [4]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

token = userdata.get('HF_TOKEN') or os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
assert token, 'Missing HF_TOKEN in Colab Secrets.'
os.environ['HF_TOKEN'] = token
os.environ['HUGGINGFACE_HUB_TOKEN'] = token
login(token=token, add_to_git_credential=False)
who = HfApi(token=token).whoami()
print('HF auth OK:', who.get('name') or who.get('email') or 'authenticated user')


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF auth OK: mshapiro123


In [5]:
%cd /content/recurrent-qwen-svgd

from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')

def restore_by_name(filename, dst, required=True, preferred_substring=None):
    dst = Path(dst)
    if dst.exists():
        print('exists:', dst)
        return dst
    matches = sorted(DRIVE_ROOT.rglob(filename))
    if preferred_substring:
        preferred = [p for p in matches if preferred_substring in str(p)]
        if preferred:
            matches = preferred
    if not matches:
        if required:
            raise FileNotFoundError(f'Could not find {filename} under {DRIVE_ROOT}')
        print('missing optional:', filename)
        return None
    src = matches[0]
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print('restored:', src, '->', dst)
    return dst

PHASE1_CKPT = 'outputs/qwen_0_5b_phase1_a100_beta008_continue_150/phase1_step_150.pt'
PHASE2_CKPT = 'outputs/qwen_0_5b_phase2_svgd_smoke25/phase2_step_25.pt'

restore_by_name('phase1_step_150.pt', PHASE1_CKPT, preferred_substring='qwen_0_5b_phase1_a100_beta008_continue_150')
restore_by_name('phase2_step_25.pt', PHASE2_CKPT, preferred_substring='qwen_0_5b_phase2_svgd_smoke25')

!ls -lh outputs/qwen_0_5b_phase1_a100_beta008_continue_150 outputs/qwen_0_5b_phase2_svgd_smoke25


/content/recurrent-qwen-svgd
Mounted at /content/drive
restored: /content/drive/MyDrive/gram-recurrent-qwen-checkpoints/phase1_step_150.pt -> outputs/qwen_0_5b_phase1_a100_beta008_continue_150/phase1_step_150.pt
restored: /content/drive/MyDrive/gram-recurrent-qwen-runs/svgd_smoke_20260618_030848/qwen_0_5b_phase2_svgd_smoke25/phase2_step_25.pt -> outputs/qwen_0_5b_phase2_svgd_smoke25/phase2_step_25.pt
outputs/qwen_0_5b_phase1_a100_beta008_continue_150:
total 12M
-rw------- 1 root root 12M Jun 18 02:35 phase1_step_150.pt

outputs/qwen_0_5b_phase2_svgd_smoke25:
total 12M
-rw------- 1 root root 12M Jun 18 03:06 phase2_step_25.pt


In [6]:
%cd /content/recurrent-qwen-svgd
!python -m pytest -q tests


/content/recurrent-qwen-svgd
.......................                                                  [100%]
23 passed in 4.05s


## Robustness Experiment

Runs three settings over seeds `0..4`:

1. raw hidden kernel, `repulsion=1.0`
2. projected 32D kernel, `repulsion=0.5` candidate-density setting
3. projected 32D kernel, `repulsion=4.0` oracle/selector setting


In [ ]:
%cd /content/recurrent-qwen-svgd

import subprocess, sys
from pathlib import Path

COMMON = [
    sys.executable, 'eval/eval_best_of_k_jsonl.py',
    '--tasks_jsonl', 'eval/smoke_exact_tasks_v2.jsonl',
    '--skip_phase1',
    '--compact',
    '--seeds', '0,1,2,3,4',
    '--phase1_checkpoint', 'outputs/qwen_0_5b_phase1_a100_beta008_continue_150/phase1_step_150.pt',
    '--phase2_checkpoint', 'outputs/qwen_0_5b_phase2_svgd_smoke25/phase2_step_25.pt',
    '--phase2_num_trajectories', '4',
    '--phase2_particle_update_mode', 'svgd',
    '--particle_init_noise', '0.05',
    '--particle_noise_every_step',
    '--particle_noise_steps', '16',
    '--svgd_projection_seed', '123',
    '--svgd_eps', '1.0',
    '--svgd_repulsion_max_norm', '1.0',
    '--temperature', '0.0',
    '--max_new_tokens', '140',
    '--dtype', 'bfloat16',
    '--adapter_dtype', 'float32',
    '--device', 'cuda',
]

RUNS = [
    ('raw_repulsion1', ['--svgd_repulsion_scale', '1.0', '--svgd_kernel_projection_dim', '0']),
    ('proj32_repulsion05', ['--svgd_repulsion_scale', '0.5', '--svgd_kernel_projection_dim', '32']),
    ('proj32_repulsion4', ['--svgd_repulsion_scale', '4.0', '--svgd_kernel_projection_dim', '32']),
]

result_paths = []
for name, extra in RUNS:
    out = f'outputs/diagnostics/v2_{name}_seeds01234.jsonl'
    result_paths.append(out)
    cmd = COMMON + ['--output_jsonl', out] + extra
    print(f'\n\n===== {name} =====')
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout)
    Path(f'{name}.log').write_text(proc.stdout, encoding='utf-8')
    assert proc.returncode == 0, f'{name} failed with exit {proc.returncode}'

print('result_paths:', result_paths)


/content/recurrent-qwen-svgd


===== raw_repulsion1 =====


In [ ]:
%cd /content/recurrent-qwen-svgd

import subprocess, sys
from pathlib import Path

paths = [
    'outputs/diagnostics/v2_raw_repulsion1_seeds01234.jsonl',
    'outputs/diagnostics/v2_proj32_repulsion05_seeds01234.jsonl',
    'outputs/diagnostics/v2_proj32_repulsion4_seeds01234.jsonl',
]

for path in paths:
    print('\n\n===== SUMMARY', path, '=====')
    proc = subprocess.run(
        [sys.executable, 'eval/summarize_best_of_k_jsonl.py', path, '--show_tasks', '--show_diagnostics'],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(proc.stdout)
    Path(Path(path).stem + '_summary.txt').write_text(proc.stdout, encoding='utf-8')
    assert proc.returncode == 0


In [ ]:
%cd /content/recurrent-qwen-svgd

import json
from collections import defaultdict
from pathlib import Path

paths = {
    'raw_repulsion1': Path('outputs/diagnostics/v2_raw_repulsion1_seeds01234.jsonl'),
    'proj32_repulsion05': Path('outputs/diagnostics/v2_proj32_repulsion05_seeds01234.jsonl'),
    'proj32_repulsion4': Path('outputs/diagnostics/v2_proj32_repulsion4_seeds01234.jsonl'),
}

print('compact comparison')
print('name,best_hits,total_tasks,candidate_hits,total_candidates,repulsion_drift,pairwise,diversity,clip_fraction')
for name, path in paths.items():
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    cells = defaultdict(list)
    for row in rows:
        cells[(row['seed'], row['task'])].append(row)
    best = sum(any(candidate['hit'] for candidate in candidates) for candidates in cells.values())
    hits = sum(bool(row['hit']) for row in rows)
    diag = defaultdict(list)
    for row in rows:
        for key, value in row.get('diagnostics', {}).items():
            if isinstance(value, (int, float)):
                diag[key].append(float(value))
    means = {key: sum(values) / len(values) for key, values in diag.items() if values}
    ratio = means.get('svgd_repulsion_rms', 0.0) / max(means.get('svgd_drift_rms', 1e-9), 1e-9)
    print(
        f"{name},{best},{len(cells)},{hits},{len(rows)},"
        f"{ratio:.6g},{means.get('svgd_pairwise_distance', float('nan')):.6g},"
        f"{means.get('trajectory_diversity', float('nan')):.6g},"
        f"{means.get('svgd_repulsion_clip_fraction', float('nan')):.6g}"
    )


In [ ]:
%cd /content/recurrent-qwen-svgd

from google.colab import drive
from pathlib import Path
import datetime as dt
import shutil

drive.mount('/content/drive')
stamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
target = Path('/content/drive/MyDrive/recurrent-qwen-svgd-runs') / f'robustness_{stamp}'
target.mkdir(parents=True, exist_ok=True)

for path in [Path('outputs/diagnostics'), *Path('.').glob('*repulsion*_summary.txt'), *Path('.').glob('*repulsion*.log')]:
    if not path.exists():
        continue
    dst = target / path.name
    if path.is_dir():
        shutil.copytree(path, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(path, dst)
    print('saved:', path, '->', dst)

print('saved run dir:', target)
!find /content/drive/MyDrive/recurrent-qwen-svgd-runs -maxdepth 2 -type f | tail -40
